# Chapter 11 — Image Segmentation

Maps to Chollet Ch.11. **Classification** = one label per *image*. **Segmentation** = one label per *pixel*.
The model outputs `(H, W, num_classes)` — a softmax at every pixel.

**Three flavors:** *semantic* (every "cat" pixel → same class), *instance* (cat 1 vs cat 2), *panoptic* (both).
This notebook does **semantic segmentation** with an **encoder–decoder** ConvNet, trained end-to-end on a
fast synthetic dataset (Oxford-IIIT Pets + Segment Anything are in the templates for Colab).

In [ ]:
import os; os.environ["KERAS_BACKEND"]="tensorflow"
import keras, numpy as np, matplotlib.pyplot as plt
from keras import layers


## 1. A synthetic segmentation dataset (runs offline, fast)
Each image has a red **circle** (class 1) and a blue **rectangle** (class 2) on a dark **background** (class 0).
The **mask** is an integer label per pixel — the segmentation equivalent of a label.

In [ ]:
H = 64
def make_data(n, seed=0):
    rng = np.random.default_rng(seed)
    X = np.zeros((n, H, H, 3), "float32"); Y = np.zeros((n, H, H, 1), "uint8")
    yy, xx = np.mgrid[0:H, 0:H]
    for i in range(n):
        X[i] = rng.uniform(0.0, 0.2, (H, H, 3))                    # dark background = class 0
        cy, cx = rng.integers(15, 49, 2); r = rng.integers(8, 16)  # circle = class 1
        circ = (yy-cy)**2 + (xx-cx)**2 <= r*r
        X[i][circ] = [0.9, 0.1, 0.1]; Y[i][circ, 0] = 1
        ry, rx = rng.integers(5, 40, 2); rh, rw = rng.integers(10, 20, 2)  # rectangle = class 2
        rect = np.zeros((H, H), bool); rect[ry:ry+rh, rx:rx+rw] = True
        X[i][rect] = [0.1, 0.1, 0.9]; Y[i][rect, 0] = 2
    return X, Y

Xtr, Ytr = make_data(800, 0); Xte, Yte = make_data(200, 1)
print("images:", Xtr.shape, " masks:", Ytr.shape, " pixel classes:", np.unique(Ytr))

fig, ax = plt.subplots(1, 4, figsize=(9, 2.4))
for k in range(2):
    ax[2*k].imshow(Xtr[k]); ax[2*k].set_title("image"); ax[2*k].axis("off")
    ax[2*k+1].imshow(Ytr[k,:,:,0], cmap="viridis"); ax[2*k+1].set_title("mask"); ax[2*k+1].axis("off")
plt.tight_layout(); plt.show()


## 2. The encoder–decoder architecture
- **Encoder (downsample)**: `Conv2D` with **strides=2** shrinks the map while deepening it → compresses
  *what* is where. We use **strided convs, not max-pooling**, because pooling throws away *where* the value
  came from — and segmentation needs precise location.
- **Decoder (upsample)**: `Conv2DTranspose` with **strides=2** *learns to upsample* back to full resolution
  (the inverse of a strided conv).
- **Head**: `Conv2D(num_classes, activation="softmax")` → a class distribution at every pixel.

Downsample 3× (64→32→16→8) then upsample 3× (8→16→32→64) to recover the original size.

In [ ]:
def get_segmentation_model(num_classes=3):
    inputs = keras.Input((H, H, 3))
    x = layers.Rescaling(1.0)(inputs)                     # already 0..1 here
    # encoder
    x = layers.Conv2D(32, 3, strides=2, activation="relu", padding="same")(x)   # 32x32
    x = layers.Conv2D(64, 3, strides=2, activation="relu", padding="same")(x)   # 16x16
    x = layers.Conv2D(128,3, strides=2, activation="relu", padding="same")(x)   # 8x8
    # decoder (learned upsampling)
    x = layers.Conv2DTranspose(128,3, strides=2, activation="relu", padding="same")(x)  # 16x16
    x = layers.Conv2DTranspose(64, 3, strides=2, activation="relu", padding="same")(x)  # 32x32
    x = layers.Conv2DTranspose(32, 3, strides=2, activation="relu", padding="same")(x)  # 64x64
    outputs = layers.Conv2D(num_classes, 3, activation="softmax", padding="same")(x)    # per-pixel softmax
    return keras.Model(inputs, outputs)

seg = get_segmentation_model()
print("input -> output:", seg.input_shape, "->", seg.output_shape)   # (.,64,64,3)->(.,64,64,3)


## 3. Train with the right metric: Intersection-over-Union (IoU)
Pixel accuracy is misleading when one class dominates (lots of background). **IoU** = overlap / union of
predicted and true regions for a class (1 = perfect, 0 = miss). Loss is per-pixel
`sparse_categorical_crossentropy`.

In [ ]:
circle_iou = keras.metrics.IoU(num_classes=3, target_class_ids=(1,), name="circle_iou",
                               sparse_y_true=True, sparse_y_pred=False)   # masks=int, preds=softmax
seg.compile("adam", "sparse_categorical_crossentropy", metrics=[circle_iou, "accuracy"])
hist = seg.fit(Xtr, Ytr, epochs=12, batch_size=32, validation_split=0.1, verbose=0)
res = seg.evaluate(Xte, Yte, verbose=0, return_dict=True)
print("test:", {k: round(v,3) for k,v in res.items()})

plt.plot(hist.history["loss"], label="train"); plt.plot(hist.history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("segmentation loss"); plt.show()


## 4. Predict a mask: `argmax` over the class axis
The prediction is `(H, W, num_classes)`; take `argmax(axis=-1)` to get the per-pixel class map.

In [ ]:
def show_prediction(idx):
    img = Xte[idx]
    pred = seg.predict(img[None], verbose=0)[0].argmax(-1)
    fig, ax = plt.subplots(1, 3, figsize=(8, 2.6))
    ax[0].imshow(img); ax[0].set_title("input"); ax[0].axis("off")
    ax[1].imshow(Yte[idx,:,:,0], cmap="viridis"); ax[1].set_title("true mask"); ax[1].axis("off")
    ax[2].imshow(pred, cmap="viridis"); ax[2].set_title("predicted mask"); ax[2].axis("off")
    plt.tight_layout(); plt.show()

for idx in [0, 5, 10]:
    show_prediction(idx)


---
# ✍️ PROBLEMS

### P1 — Per-class IoU
Add IoU metrics for **all three** classes (background, circle, rectangle) and report each after training.
Which class is hardest? Why might background be easiest?

In [ ]:
# TODO


### P2 — Max-pooling vs strided conv
Build a second model that downsamples with `MaxPooling2D` instead of strided conv (and `UpSampling2D` to go
back up). Compare IoU and the visual sharpness of predicted masks. Does losing location info hurt?

In [ ]:
# TODO


### P3 — Skip connections (toward U-Net)
Add skip connections: concatenate each encoder feature map with the matching decoder feature map of the same
size (`layers.Concatenate`). This is the **U-Net** idea. Does it sharpen the masks / raise IoU?

In [ ]:
# TODO


### P4 — Harder shapes
Extend `make_data` with a third object (e.g. a triangle = class 3), overlapping shapes, and noise. Retrain
(num_classes=4) and report per-class IoU. Where does the model struggle?

In [ ]:
# TODO


---
# 📋 TEMPLATES

### T1 — Encoder–decoder segmentation model

In [ ]:
from keras import layers
import keras
def get_segmentation_model(img_size, num_classes):
    inputs = keras.Input(img_size + (3,))
    x = layers.Rescaling(1./255)(inputs)
    for f in (64, 128, 256):                              # encoder: strided conv (keeps location)
        x = layers.Conv2D(f, 3, strides=2, activation="relu", padding="same")(x)
        x = layers.Conv2D(f, 3, activation="relu", padding="same")(x)
    for f in (256, 128, 64):                              # decoder: learned upsampling
        x = layers.Conv2DTranspose(f, 3, activation="relu", padding="same")(x)
        x = layers.Conv2DTranspose(f, 3, strides=2, activation="relu", padding="same")(x)
    outputs = layers.Conv2D(num_classes, 3, activation="softmax", padding="same")(x)
    return keras.Model(inputs, outputs)
# compile: loss="sparse_categorical_crossentropy" (integer masks); metric: keras.metrics.IoU(...)


### T2 — Oxford-IIIT Pets (real data, Colab)

In [ ]:
# !wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz && tar -xf images.tar.gz
# !wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz && tar -xf annotations.tar.gz
# masks in annotations/trimaps/*.png have values {1,2,3}; subtract 1 -> {0,1,2}
# load each image to (200,200,3) float and mask to (200,200,1) uint8, then fit the T1 model.


### T3 — Segment Anything (SAM), pretrained, via KerasHub (Colab)

In [ ]:
# import keras_hub
# model = keras_hub.models.ImageSegmenter.from_preset("sam_huge_sa1b")   # ~641M params
# prompt with a point or a box inside the object you want; SAM returns its mask. No fine-tuning needed.


---
### ✅ Checklist
- [ ] Explain segmentation vs classification (per-pixel softmax, output (H,W,C)).
- [ ] Know semantic vs instance vs panoptic.
- [ ] Build an encoder–decoder; explain strided conv (keeps location) vs max-pool, and Conv2DTranspose upsampling.
- [ ] Train with sparse_categorical_crossentropy + IoU; predict a mask via argmax.
- [ ] Know U-Net skip connections and that SAM exists for zero-shot segmentation.

**Next: Chapter 12** — *Object detection* (bounding boxes + classes). Say "Chapter 12".